In [34]:
tabla ='qtiaa10'
col_tabla = 'solopefec'
essi='essi_dat_cqx006_etl'
col_essi='sol_fec'

In [35]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [36]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [37]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [38]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"
# c1= text(query)
# connection2.execute(c1)

#INICIO DEL ESSI

In [39]:
fecha = '01/05/23'

In [40]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPEINFANE,
d1.INFOPEANESECUE,
d1.INFOPEANEANECOD,
d1.INFOPEANEESTREGCOD,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM
from sgss.{tabla} d1
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM

where a1.{col_tabla}>='{fecha}'
"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [41]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32959 entries, 0 to 32958
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  32959 non-null  object        
 1   infopecenasicod     32959 non-null  object        
 2   infopesolopenum     32959 non-null  int64         
 3   infopeinfane        32959 non-null  int64         
 4   infopeanesecue      32959 non-null  int64         
 5   infopeaneanecod     32959 non-null  object        
 6   infopeaneestregcod  32959 non-null  object        
 7   solopefec           32959 non-null  datetime64[ns]
 8   solopeactmednum     32959 non-null  int64         
 9   solopebuspacsecnum  32959 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 2.5+ MB


In [42]:
base1.columns

Index(['infopeoricenasicod', 'infopecenasicod', 'infopesolopenum',
       'infopeinfane', 'infopeanesecue', 'infopeaneanecod',
       'infopeaneestregcod', 'solopefec', 'solopeactmednum',
       'solopebuspacsecnum'],
      dtype='object')

In [43]:
base1 = base1.rename(columns={
    'infopeoricenasicod': 'ori_cas',
    'infopecenasicod': 'cod_cas',
    'infopesolopenum': 'sol_num',
    'infopeinfane': 'inf_ane',
    'infopeanesecue': 'sec_ane',
    'infopeaneanecod': 'cod_ane',
    'infopeaneestregcod': 'est_reg',
    'solopefec': 'sol_fec',
    'solopeactmednum': 'act_med',
    'solopebuspacsecnum': 'pac_sec'
})

In [44]:
base1.shape

(32959, 10)

In [45]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [46]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ori_cas  10 non-null     object 
 1   cod_cas  10 non-null     object 
 2   des_cas  10 non-null     object 
 3   cod_red  10 non-null     object 
 4   des_red  10 non-null     object 
 5   sol_num  10 non-null     float64
 6   inf_ane  10 non-null     float64
 7   sec_ane  10 non-null     float64
 8   cod_ane  10 non-null     object 
 9   des_ane  10 non-null     object 
 10  est_reg  10 non-null     object 
 11  sol_fec  10 non-null     object 
 12  act_med  10 non-null     float64
 13  pac_sec  10 non-null     float64
dtypes: float64(5), object(9)
memory usage: 1.2+ KB


#TRAEMOS TODOS LOS MAESTROS

In [47]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

cqxaneste= pd.read_sql_query(f"SELECT cod_ane,des_ane FROM dim_cqxaneste", con=connection2)
cqxaneste['cod_ane']=cqxaneste['cod_ane'].str.strip()

In [48]:
a=base1.copy()

In [49]:
# base1=a

In [50]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32959 entries, 0 to 32958
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  32959 non-null  object        
 1   cod_cas  32959 non-null  object        
 2   sol_num  32959 non-null  int64         
 3   inf_ane  32959 non-null  int64         
 4   sec_ane  32959 non-null  int64         
 5   cod_ane  32959 non-null  object        
 6   est_reg  32959 non-null  object        
 7   sol_fec  32959 non-null  datetime64[ns]
 8   act_med  32959 non-null  int64         
 9   pac_sec  32959 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 2.5+ MB


In [51]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(32959, 13)

In [52]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32959 entries, 0 to 32958
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  32959 non-null  object        
 1   cod_cas  32959 non-null  object        
 2   sol_num  32959 non-null  int64         
 3   inf_ane  32959 non-null  int64         
 4   sec_ane  32959 non-null  int64         
 5   cod_ane  32959 non-null  object        
 6   est_reg  32959 non-null  object        
 7   sol_fec  32959 non-null  datetime64[ns]
 8   act_med  32959 non-null  int64         
 9   pac_sec  32959 non-null  int64         
 10  des_cas  32959 non-null  object        
 11  cod_red  32959 non-null  object        
 12  des_red  32959 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(7)
memory usage: 3.5+ MB


In [53]:
#des_ane
base1['cod_ane']=base1['cod_ane'].str.strip()
base1=pd.merge(base1,cqxaneste,how='left',on='cod_ane')
base1.shape

(32959, 14)

In [54]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32959 entries, 0 to 32958
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  32959 non-null  object        
 1   cod_cas  32959 non-null  object        
 2   sol_num  32959 non-null  int64         
 3   inf_ane  32959 non-null  int64         
 4   sec_ane  32959 non-null  int64         
 5   cod_ane  32959 non-null  object        
 6   est_reg  32959 non-null  object        
 7   sol_fec  32959 non-null  datetime64[ns]
 8   act_med  32959 non-null  int64         
 9   pac_sec  32959 non-null  int64         
 10  des_cas  32959 non-null  object        
 11  cod_red  32959 non-null  object        
 12  des_red  32959 non-null  object        
 13  des_ane  32959 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(8)
memory usage: 3.8+ MB


In [55]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'sol_num',
       'inf_ane', 'sec_ane', 'cod_ane', 'des_ane', 'est_reg', 'sol_fec',
       'act_med', 'pac_sec'],
      dtype='object')

In [56]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [57]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [58]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [59]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32959 entries, 0 to 32958
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   cod_cas  32959 non-null  object        
 1   est_reg  32959 non-null  object        
 2   sol_fec  32959 non-null  datetime64[ns]
 3   cod_red  32959 non-null  object        
 4   des_red  32959 non-null  object        
 5   sol_num  32959 non-null  int64         
 6   act_med  32959 non-null  int64         
 7   sec_ane  32959 non-null  int64         
 8   ori_cas  32959 non-null  object        
 9   pac_sec  32959 non-null  int64         
 10  des_cas  32959 non-null  object        
 11  inf_ane  32959 non-null  int64         
 12  des_ane  32959 non-null  object        
 13  cod_ane  32959 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(8)
memory usage: 3.8+ MB


In [60]:
base.head()

,cod_cas,est_reg,sol_fec,cod_red,des_red,sol_num,act_med,sec_ane,ori_cas,pac_sec,des_cas,inf_ane,des_ane,cod_ane
0,005,1,2023-05-09,05,RED PRESTACIONAL SABOGAL ...,3828,6708687,1,1,23685498,H.N. ALBERTO SABOGAL SOLOGUREN,1,GENERAL INHALATORIA ...,101
1,006,1,2029-07-10,07,RED PRESTACIONAL REBAGLIATI ...,5182,6799154,1,1,17437280,H.III SUAREZ-ANGAMOS,1,ANESTESIA GENERAL CON INTUBACION ENDOTRAQUEAL ...,104
2,191,1,2023-05-17,12,RED ASISTENCIAL CAJAMARCA ...,10908,1266549,1,1,24041461,H.II CAJAMARCA,1,BLOQUEO SUBDURAL SIMPLE - RAQUIDEA ...,407
3,001,1,2023-06-17,07,RED PRESTACIONAL REBAGLIATI ...,58544,10551931,1,1,10290180,H.N. EDGARDO REBAGLIATI MARTINS,1,LOCAL + SEDACION ...,202
4,001,1,2023-08-09,07,RED PRESTACIONAL REBAGLIATI ...,71893,10456944,1,1,8120560,H.N. EDGARDO REBAGLIATI MARTINS,1,LOCAL + BLOQUEO ...,203


In [61]:
base.columns

Index(['cod_cas', 'est_reg', 'sol_fec', 'cod_red', 'des_red', 'sol_num',
       'act_med', 'sec_ane', 'ori_cas', 'pac_sec', 'des_cas', 'inf_ane',
       'des_ane', 'cod_ane'],
      dtype='object')

In [62]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [63]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

3959

In [64]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 6.794 segundos


In [65]:
connection1.close()
connection2.close()
connection3.close()

In [66]:
engine1.dispose()
engine2.dispose()
engine3.dispose()